In [ ]:
from notebook_utils import cd2parent
cd2parent()

In [5]:
import os
import re

import pandas as pd
import numpy as np

from src.paths import smi2safepath, parse_params

In [6]:
reactions = pd.read_csv('data/reactions/raw/reactions.csv')
reactions

,reactant,product,carbene
0,COc1cc(OC)c([S+](c2ccc(Cl)cc2)[C-]2COC2)c(OC)c1,COc1cc(OC)c(Sc2ccc(Cl)cc2)c(OC)c1,[C]1COC1
1,COc1cc(OC)c([S+](c2ccccc2)[C-]2CCC2)c(OC)c1,COc1cc(OC)c(Sc2ccccc2)c(OC)c1,[C]1CCC1
2,COc1cc(OC)c([S+](c2ccc(Cl)cc2)[C-]2CN(S(=O)(=O...,COc1cc(OC)c(Sc2ccc(Cl)cc2)c(OC)c1,Cc1ccc(S(=O)(=O)N2C[C]C2)cc1


In [7]:
hartree_divisor = 627.5095

def G_from_smi(smi, dir_dft = 'data/reactions/DFT/method=m062x|basis=def2-svp|freq=True|solvent=thf|tightscf=True'):
	safepath = smi2safepath(smi)
	# patt =r'G-E\(el\)[\s\.\+\-0-9]+Eh\s+(?P<g>[-\+\.0-9]+)\s{1}kcal\/mol'
	patt =r'Final Gibbs free energy\s+\.{3}\s+(?P<g>[-\+\.0-9]+)\s{1}Eh'
	path = os.path.join(dir_dft, safepath, 'file.out')

	if not os.path.exists(path):
		return np.nan

	with open(path,'r') as f:
		value = re.search(patt, f.read())

	if value is None:
		return np.nan
	
	gibbs_hartree = float(value['g'])

	return gibbs_hartree*hartree_divisor

In [8]:
params = parse_params('method=m062x|basis=def2-svp|freq=True|solvent=thf|tightscf=True')
params

{'method': 'm062x',
 'basis': 'def2-svp',
 'freq': 'True',
 'solvent': 'thf',
 'tightscf': 'True'}

In [9]:
gibbs = reactions.map(G_from_smi)
gibbs

,reactant,product,carbene
0,-1.163870e+06,-1.043686e+06,-120170.325015
1,-8.530365e+05,-7.553709e+05,-97649.727640
2,-1.664850e+06,-1.043686e+06,-621150.153566


In [10]:
delta_gibbs = gibbs['carbene'] + gibbs['product'] - gibbs['reactant']
delta_gibbs

0    13.625145
1    15.895192
2    13.708008
dtype: float64

# Drawing

In [11]:
from rdkit import Chem

from src.utils import drawMol

import svgutils.transform as sg
from svgutils.compose import Unit

In [12]:
for smi in reactions.values.flatten():
	mol = Chem.MolFromSmiles(smi)

	svg = drawMol(mol=mol ,remove_background=True)
	with open(f'figures/reactions/{smi}.svg','w') as f:
		f.write(svg)

In [13]:
# Loading elements
elements_svg = reactions.map(lambda x: sg.fromfile(f'figures/reactions/{x}.svg'))
heights_svg = elements_svg.map(lambda x: int(re.search(r'[0-9]+',x.height).group()))
widths_svg = elements_svg.map(lambda x: int(re.search(r'[0-9]+',x.width).group()))

# Grid parameters
legend_height = 30  # legend height
padding_legend = 10

padding = 20        # padding between elements
arrow_length = 160
plus_width = 40

widths_cols = widths_svg.max(axis=0).values
heights_rows = heights_svg.max(axis=1).values

# Number of rows and canvas size
canvas_width = widths_cols.sum() + arrow_length + padding*4 + plus_width
canvas_height = heights_rows.sum() + (legend_height+padding_legend)*3 + padding*2

x_delta = widths_cols[0] + padding + arrow_length/2
x_plus = widths_cols[:2].sum() + padding*3 + arrow_length + plus_width/2

In [14]:
def make_arrow(x, y, dx=100, dy=0, color="black", stroke_width=2):
    arrow_svg = f"""
    <svg xmlns="http://www.w3.org/2000/svg">
      <defs>
        <marker id="arrowhead" markerWidth="7" markerHeight="7" 
                refX="7" refY="3.5" orient="auto">
          <polygon points="0 0, 7 3.5, 0 7" fill="{color}" />
        </marker>
      </defs>
      <line x1="{x}" y1="{y}" x2="{x+dx}" y2="{y+dy}" 
            stroke="{color}" stroke-width="{stroke_width}" 
            marker-end="url(#arrowhead)" />
    </svg>
    """
    return sg.fromstring(arrow_svg).getroot()

In [15]:
label_structures =[
    'reactant',
    'product',
    'carbene',
]

In [16]:
# Crea una figura SVG finale
final_fig = sg.SVGFigure()
final_fig.set_size((str(canvas_width)+'px', str(canvas_height)+'px'))
elements = []

for idx_row in range(3):

	center_y = heights_rows[:idx_row].sum() + heights_rows[idx_row]/2
	center_y += (legend_height+padding_legend)*(idx_row) + padding*(idx_row)

	delta_value = delta_gibbs[idx_row]

	delta_text = sg.TextElement(
		x_delta,
		center_y - 5,
		f'ΔG : {delta_value:.2f} kcal/mol',
		size=12,
		anchor="middle"
	)

	plus_text = sg.TextElement(
		x_plus,
		center_y,
		f'+',
		size=15,
		anchor="middle"
	)

	arrow = make_arrow(
		x_delta - arrow_length/2,
		center_y,
		arrow_length,
	)

	elements.extend([delta_text, plus_text, arrow])

	for idx_col in range(3):
		root = elements_svg.iloc[idx_row,idx_col].getroot()

		center_x = widths_cols[:idx_col].sum() + widths_cols[idx_col]/2
		if idx_col > 0:
			center_x += padding*2 + arrow_length

		if idx_col > 1:
			center_x+= padding*2 + plus_width
			
		x,y = (center_x - widths_svg.iloc[idx_row, idx_col]/2),(center_y - heights_svg.iloc[idx_row, idx_col]/2)

		root.moveto(x, y)

		gibbs_value = gibbs.iloc[idx_row,idx_col]
		# legend = f'G : {gibbs_value:.2f} kcal/mol'
		legend = label_structures[idx_col] + '_' + str(idx_row+1)

		# Crea la legenda sotto l'immagine
		text = sg.TextElement(
			center_x,
			center_y + heights_rows[idx_row]/2 + padding_legend,
			legend,
			size=12,
			anchor="middle",
		)

		elements.extend([root, text])

In [17]:
final_fig.append(elements)
final_fig.save("figures/reactions/carbene_reactions.svg")

# 3D images

In [ ]:
from rdkit import Chem

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG=True  # Use higher quality images for molecules
import py3Dmol

In [19]:
def molBlock_from_smi(smi, dir_mol = 'data/reactions/opt_mol/method=m062x|basis=def2-svp|freq=True|solvent=thf|tightscf=True', removeHs = True):
	path = os.path.join(dir_mol, f'{smi}.mol')


	with open(path) as f:
		mol = Chem.MolFromMolBlock(f.read(), removeHs=removeHs)

	Chem.RemoveHs(mol)

	return Chem.MolToMolBlock(mol)

In [ ]:
view = py3Dmol.view(
    data=molBlock_from_smi(reactions.iloc[0,2]),  # Convert the RDKit molecule for py3Dmol
    style={
        "stick": {"hideHydrogens": True},
        # 'line':{},
        # "sphere": {"scale": 0.3},
    }
)
view.zoomTo()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.